# 10 — Neuromaps BrainSMASH: PC1 Loadings vs Gene Expression Gradients

Tests whether the spatial pattern of PC1 loadings (brain status and brain pace) aligns with AHBA gene expression gradients using BrainSMASH spatial autocorrelation-preserving null tests.

**Inputs:**
- PC1 loadings from `08_pca_brain_scores.ipynb` (cross-sectional V1/V2; longitudinal V1_V2/V2_V3)
- AHBA gene expression gradients (Deary et al. top-8k genes, HCP MMP1.0 parcellation)
- HCP MMP1.0 and Desikan–Killiany (DK) surface atlases
- fsaverage5 surface annotations for parcel centroid geometry

**Output:** `brainsmash_results.csv` — r, p_raw, p_fdr, reject for all 12 tests (4 analyses × 3 gene PCs), FDR corrected within group (cross-sectional / longitudinal).

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
# AHBA gene expression gradients (downloaded from GitHub at runtime)
GENE_GRADIENTS_URL = (
    "https://raw.githubusercontent.com/richardajdear/AHBA_gradients/"
    "ab7939cf1811cba6296b882e35e60b09fed7d653/outputs/"
    "ahba_dme_hcp_top8kgenes_scores.csv"
)

# Surface atlas files
HCP_ATLAS        = 'data/atlases/HCP_MMP1.32k_fs_LR.dlabel.nii'
DK_ATLAS         = 'data/atlases/fsaverage.aparc.32k_fs_LR.dlabel.nii'
FSAVG5_LABEL_DIR = '~/freesurfer/subjects/fsaverage5/label'

# PC1 loadings directories (output of 08_pca_brain_scores.ipynb)
PCA_LONG_DIR = 'outputs/pca/longitudinal'
PCA_CS_DIR   = 'outputs/pca/cross_sectional'

OUTPUT_DIR = 'outputs/neuromaps'

N_PERM = 10000
SEED   = 42

# Analyses: (label, loadings_file, FDR_group)
# FDR correction is applied within each group independently.
ANALYSES = [
    ('V1_status',  f'{PCA_CS_DIR}/PC1_Loadings_V1_healthy.csv',     'cross_sectional'),
    ('V2_status',  f'{PCA_CS_DIR}/PC1_Loadings_V2_healthy.csv',     'cross_sectional'),
    ('V1V2_pace',  f'{PCA_LONG_DIR}/PC1_Loadings_V1_V2_healthy.csv', 'longitudinal'),
    ('V2V3_pace',  f'{PCA_LONG_DIR}/PC1_Loadings_V2_V3_healthy.csv', 'longitudinal'),
]
GENE_PCS = ['PC1', 'PC2', 'PC3']

In [ ]:
import os
import numpy as np
import pandas as pd
import nibabel as nib
import nilearn.datasets as nilearn_datasets
import nilearn.surface  as nilearn_surface
from brainsmash.mapgen.base import Base
from statsmodels.stats.multitest import multipletests

os.makedirs(OUTPUT_DIR, exist_ok=True)
FSAVG5_LABEL_DIR = os.path.expanduser(FSAVG5_LABEL_DIR)

## Step 1 — Load AHBA gene expression gradients and parcellate into DK atlas

Gene expression PCs (C1/C2/C3) are defined in HCP MMP1.0 space. We project them into
Desikan–Killiany (DK) parcels by averaging vertex values within each DK ROI.

In [ ]:
# Load AHBA gene gradient scores (HCP MMP1.0 parcellation)
df_genes = pd.read_csv(GENE_GRADIENTS_URL)

# Load HCP MMP1.0 atlas and project gene PCs to vertex space (bilateral)
hcp_atlas  = nib.load(HCP_ATLAS)
atlas_data = hcp_atlas.get_fdata()[0]

Gene_PC1_vert = atlas_data * np.nan
Gene_PC2_vert = atlas_data * np.nan
Gene_PC3_vert = atlas_data * np.nan

for _, row in df_genes.iterrows():
    mask = (atlas_data == row.id) | (atlas_data == (row.id + 180))
    Gene_PC1_vert[mask] = row.C1
    Gene_PC2_vert[mask] = row.C2
    Gene_PC3_vert[mask] = row.C3

# Load DK atlas and aggregate gene PCs into DK parcels
dk_atlas  = nib.load(DK_ATLAS)
dk_data   = dk_atlas.get_fdata()[0]
dk_labels = list(dk_atlas.header.get_index_map(0).named_maps)[0].label_table

dk_results = []
for idx, label_obj in dk_labels.items():
    name = label_obj.label
    if name in ('???', 'unknown', 'Medial_Wall', 'Background'):
        continue
    mask = (dk_data == idx)
    if mask.any():
        dk_results.append({
            'ROI': name.replace('_', '.', 1),  # L_bankssts → L.bankssts
            'PC1': np.nanmean(Gene_PC1_vert[mask]),
            'PC2': np.nanmean(Gene_PC2_vert[mask]),
            'PC3': np.nanmean(Gene_PC3_vert[mask]),
        })

df_dk = pd.DataFrame(dk_results)
df_dk['hemi']   = np.where(df_dk['ROI'].str.startswith('L.'), 'lh', 'rh')
df_dk['Region'] = df_dk['ROI'].str[2:]

df_dk_lh = df_dk[df_dk['hemi'] == 'lh'].set_index('Region')
df_dk_rh = df_dk[df_dk['hemi'] == 'rh'].set_index('Region')

print(f'DK parcels: {len(df_dk_lh)} LH + {len(df_dk_rh)} RH')

## Step 2 — Load DK surface geometry for BrainSMASH

BrainSMASH needs parcel centroids (coordinates) to model spatial autocorrelation.

In [ ]:
DK_ORDER = [
    'bankssts', 'caudalanteriorcingulate', 'caudalmiddlefrontal', 'cuneus',
    'entorhinal', 'frontalpole', 'fusiform', 'inferiorparietal',
    'inferiortemporal', 'insula', 'isthmuscingulate', 'lateraloccipital',
    'lateralorbitofrontal', 'lingual', 'medialorbitofrontal', 'middletemporal',
    'paracentral', 'parahippocampal', 'parsopercularis', 'parsorbitalis',
    'parstriangularis', 'pericalcarine', 'postcentral', 'posteriorcingulate',
    'precentral', 'precuneus', 'rostralanteriorcingulate', 'rostralmiddlefrontal',
    'superiorfrontal', 'superiorparietal', 'superiortemporal', 'supramarginal',
    'temporalpole', 'transversetemporal',
]

# Load fsaverage5 DK annotations
dk_l, _, names_l = nib.freesurfer.read_annot(os.path.join(FSAVG5_LABEL_DIR, 'lh.aparc.annot'))
dk_r, _, names_r = nib.freesurfer.read_annot(os.path.join(FSAVG5_LABEL_DIR, 'rh.aparc.annot'))
names_l = [n.decode() if isinstance(n, bytes) else n for n in names_l]
names_r = [n.decode() if isinstance(n, bytes) else n for n in names_r]
name_map_l = {name: i for i, name in enumerate(names_l)}
name_map_r = {name: i for i, name in enumerate(names_r)}

# Load fsaverage5 mesh coordinates
fsavg = nilearn_datasets.fetch_surf_fsaverage(mesh='fsaverage5')
coords_lh = np.asarray(nilearn_surface.load_surf_mesh(fsavg['pial_left']).coordinates)
coords_rh = np.asarray(nilearn_surface.load_surf_mesh(fsavg['pial_right']).coordinates)

assert dk_l.shape[0] == coords_lh.shape[0]
assert dk_r.shape[0] == coords_rh.shape[0]

# Compute parcel centroids (done once; same for all analyses)
def parcel_centroids(dk_annot, coords, name_map, region_order):
    kept, centroid_list = [], []
    for region in region_order:
        lab = name_map.get(region)
        if lab is None:
            continue
        mask = (dk_annot == lab)
        if mask.any():
            centroid_list.append(coords[mask].mean(axis=0))
            kept.append(region)
    return kept, np.vstack(centroid_list)

regions_lh, cents_lh = parcel_centroids(dk_l, coords_lh, name_map_l, DK_ORDER)
regions_rh, cents_rh = parcel_centroids(dk_r, coords_rh, name_map_r, DK_ORDER)
coords_bi_all = np.vstack([cents_lh, cents_rh])

print(f'Parcel centroids: {len(regions_lh)} LH + {len(regions_rh)} RH')

## Step 3 — BrainSMASH spatial null tests

For each analysis (4 PC1 loading maps) × gene expression PC (3), we:
1. Load ABCD PC1 loadings and align to DK parcels
2. Build bilateral loading vector `x` and gene PC vector `y`
3. Run BrainSMASH: generate spatially autocorrelated surrogates of `y`, compute null r distribution
4. Compute observed r and permutation p-value

In [ ]:
ABCD_TO_DK = {
    # Left cortical
    'mr_y_smri__vol__dsk__bstmps__lh_sum': 'L.bankssts',
    'mr_y_smri__vol__dsk__cac__lh_sum':    'L.caudalanteriorcingulate',
    'mr_y_smri__vol__dsk__cmfrt__lh_sum':  'L.caudalmiddlefrontal',
    'mr_y_smri__vol__dsk__cn__lh_sum':     'L.cuneus',
    'mr_y_smri__vol__dsk__er__lh_sum':     'L.entorhinal',
    'mr_y_smri__vol__dsk__ff__lh_sum':     'L.fusiform',
    'mr_y_smri__vol__dsk__iprt__lh_sum':   'L.inferiorparietal',
    'mr_y_smri__vol__dsk__itmp__lh_sum':   'L.inferiortemporal',
    'mr_y_smri__vol__dsk__ic__lh_sum':     'L.isthmuscingulate',
    'mr_y_smri__vol__dsk__locc__lh_sum':   'L.lateraloccipital',
    'mr_y_smri__vol__dsk__lobfrt__lh_sum': 'L.lateralorbitofrontal',
    'mr_y_smri__vol__dsk__lg__lh_sum':     'L.lingual',
    'mr_y_smri__vol__dsk__mobfrt__lh_sum': 'L.medialorbitofrontal',
    'mr_y_smri__vol__dsk__mtmp__lh_sum':   'L.middletemporal',
    'mr_y_smri__vol__dsk__ph__lh_sum':     'L.parahippocampal',
    'mr_y_smri__vol__dsk__pactr__lh_sum':  'L.paracentral',
    'mr_y_smri__vol__dsk__pop__lh_sum':    'L.parsopercularis',
    'mr_y_smri__vol__dsk__pob__lh_sum':    'L.parsorbitalis',
    'mr_y_smri__vol__dsk__ptg__lh_sum':    'L.parstriangularis',
    'mr_y_smri__vol__dsk__pcc__lh_sum':    'L.pericalcarine',
    'mr_y_smri__vol__dsk__poctr__lh_sum':  'L.postcentral',
    'mr_y_smri__vol__dsk__pcg__lh_sum':    'L.posteriorcingulate',
    'mr_y_smri__vol__dsk__prctr__lh_sum':  'L.precentral',
    'mr_y_smri__vol__dsk__prcn__lh_sum':   'L.precuneus',
    'mr_y_smri__vol__dsk__rac__lh_sum':    'L.rostralanteriorcingulate',
    'mr_y_smri__vol__dsk__rmfrt__lh_sum':  'L.rostralmiddlefrontal',
    'mr_y_smri__vol__dsk__sfrt__lh_sum':   'L.superiorfrontal',
    'mr_y_smri__vol__dsk__sprt__lh_sum':   'L.superiorparietal',
    'mr_y_smri__vol__dsk__stmp__lh_sum':   'L.superiortemporal',
    'mr_y_smri__vol__dsk__sm__lh_sum':     'L.supramarginal',
    'mr_y_smri__vol__dsk__pfrt__lh_sum':   'L.frontalpole',
    'mr_y_smri__vol__dsk__ptmp__lh_sum':   'L.temporalpole',
    'mr_y_smri__vol__dsk__ttmp__lh_sum':   'L.transversetemporal',
    'mr_y_smri__vol__dsk__ins__lh_sum':    'L.insula',
    # Right cortical
    'mr_y_smri__vol__dsk__bstmps__rh_sum': 'R.bankssts',
    'mr_y_smri__vol__dsk__cac__rh_sum':    'R.caudalanteriorcingulate',
    'mr_y_smri__vol__dsk__cmfrt__rh_sum':  'R.caudalmiddlefrontal',
    'mr_y_smri__vol__dsk__cn__rh_sum':     'R.cuneus',
    'mr_y_smri__vol__dsk__er__rh_sum':     'R.entorhinal',
    'mr_y_smri__vol__dsk__ff__rh_sum':     'R.fusiform',
    'mr_y_smri__vol__dsk__iprt__rh_sum':   'R.inferiorparietal',
    'mr_y_smri__vol__dsk__itmp__rh_sum':   'R.inferiortemporal',
    'mr_y_smri__vol__dsk__ic__rh_sum':     'R.isthmuscingulate',
    'mr_y_smri__vol__dsk__locc__rh_sum':   'R.lateraloccipital',
    'mr_y_smri__vol__dsk__lobfrt__rh_sum': 'R.lateralorbitofrontal',
    'mr_y_smri__vol__dsk__lg__rh_sum':     'R.lingual',
    'mr_y_smri__vol__dsk__mobfrt__rh_sum': 'R.medialorbitofrontal',
    'mr_y_smri__vol__dsk__mtmp__rh_sum':   'R.middletemporal',
    'mr_y_smri__vol__dsk__ph__rh_sum':     'R.parahippocampal',
    'mr_y_smri__vol__dsk__pactr__rh_sum':  'R.paracentral',
    'mr_y_smri__vol__dsk__pop__rh_sum':    'R.parsopercularis',
    'mr_y_smri__vol__dsk__pob__rh_sum':    'R.parsorbitalis',
    'mr_y_smri__vol__dsk__ptg__rh_sum':    'R.parstriangularis',
    'mr_y_smri__vol__dsk__pcc__rh_sum':    'R.pericalcarine',
    'mr_y_smri__vol__dsk__poctr__rh_sum':  'R.postcentral',
    'mr_y_smri__vol__dsk__pcg__rh_sum':    'R.posteriorcingulate',
    'mr_y_smri__vol__dsk__prctr__rh_sum':  'R.precentral',
    'mr_y_smri__vol__dsk__prcn__rh_sum':   'R.precuneus',
    'mr_y_smri__vol__dsk__rac__rh_sum':    'R.rostralanteriorcingulate',
    'mr_y_smri__vol__dsk__rmfrt__rh_sum':  'R.rostralmiddlefrontal',
    'mr_y_smri__vol__dsk__sfrt__rh_sum':   'R.superiorfrontal',
    'mr_y_smri__vol__dsk__sprt__rh_sum':   'R.superiorparietal',
    'mr_y_smri__vol__dsk__stmp__rh_sum':   'R.superiortemporal',
    'mr_y_smri__vol__dsk__sm__rh_sum':     'R.supramarginal',
    'mr_y_smri__vol__dsk__pfrt__rh_sum':   'R.frontalpole',
    'mr_y_smri__vol__dsk__ptmp__rh_sum':   'R.temporalpole',
    'mr_y_smri__vol__dsk__ttmp__rh_sum':   'R.transversetemporal',
    'mr_y_smri__vol__dsk__ins__rh_sum':    'R.insula',
}


def load_loadings(path):
    """Load ABCD PC1 loadings CSV and return (lh_df, rh_df) aligned to DK_ORDER."""
    df = pd.read_csv(path)
    df = df.rename(columns={df.columns[0]: 'ROI'})
    df['ROI']    = df['ROI'].replace(ABCD_TO_DK)
    df['hemi']   = np.where(df['ROI'].str.startswith('L.'), 'lh',
                   np.where(df['ROI'].str.startswith('R.'), 'rh', 'sub'))
    df           = df[df['hemi'].isin(['lh', 'rh'])].copy()
    df['Region'] = df['ROI'].str[2:]
    lh = df[df['hemi'] == 'lh'].set_index('Region').reindex(DK_ORDER)
    rh = df[df['hemi'] == 'rh'].set_index('Region').reindex(DK_ORDER)
    return lh, rh


def run_brainsmash(x, y, coords, n_perm=10000, seed=42):
    """BrainSMASH spatial null test; returns (r_observed, p_value)."""
    D          = np.linalg.norm(coords[:, None, :] - coords[None, :, :], axis=-1)
    gen        = Base(x=y, D=D, seed=seed, resample=True)
    surrogates = gen(n=n_perm)
    r_obs      = np.corrcoef(x, y)[0, 1]
    r_nulls    = np.array([np.corrcoef(x, surrogates[i])[0, 1] for i in range(n_perm)])
    p_val      = (np.sum(np.abs(r_nulls) >= np.abs(r_obs)) + 1) / (n_perm + 1)
    return float(r_obs), float(p_val)


results = []

for label, loadings_path, group in ANALYSES:
    print(f'\n--- {label} ({group}) ---')
    lh, rh = load_loadings(loadings_path)

    # Regions present in both loadings and centroid lists
    rh_use = [r for r in regions_lh if r in lh.index and pd.notna(lh.loc[r, 'PC1_Loading'])]
    rr_use = [r for r in regions_rh if r in rh.index and pd.notna(rh.loc[r, 'PC1_Loading'])]
    lh_idx = [regions_lh.index(r) for r in rh_use]
    rh_idx = [regions_rh.index(r) for r in rr_use]
    coords_bi = np.vstack([cents_lh[lh_idx], cents_rh[rh_idx]])

    x_bi = np.concatenate([
        lh.loc[rh_use, 'PC1_Loading'].astype(float).values,
        rh.loc[rr_use, 'PC1_Loading'].astype(float).values,
    ])

    for gene_pc in GENE_PCS:
        y_bi = np.concatenate([
            df_dk_lh.loc[rh_use, gene_pc].astype(float).values,
            df_dk_rh.loc[rr_use, gene_pc].astype(float).values,
        ])
        r, p = run_brainsmash(x_bi, y_bi, coords_bi, N_PERM, SEED)
        print(f'  vs {gene_pc}: r={r:.3f}  p={p:.5f}')
        results.append({
            'analysis': label, 'group': group, 'gene_pc': gene_pc,
            'r': r, 'p_raw': p,
        })

df_results = pd.DataFrame(results)

## Step 4 — FDR correction and results summary

Benjamini–Hochberg FDR correction applied **within each group** (6 tests per group):  
- Cross-sectional: V1_status × {PC1, PC2, PC3} + V2_status × {PC1, PC2, PC3}  
- Longitudinal: V1V2_pace × {PC1, PC2, PC3} + V2V3_pace × {PC1, PC2, PC3}

In [ ]:
df_results['p_fdr']  = np.nan
df_results['reject'] = False

for grp in df_results['group'].unique():
    mask = df_results['group'] == grp
    reject, p_fdr, _, _ = multipletests(df_results.loc[mask, 'p_raw'], method='fdr_bh')
    df_results.loc[mask, 'p_fdr']  = p_fdr
    df_results.loc[mask, 'reject'] = reject

print(df_results[['analysis', 'gene_pc', 'r', 'p_raw', 'p_fdr', 'reject']]
      .to_string(index=False, float_format='{:.4f}'.format))

out_path = os.path.join(OUTPUT_DIR, 'brainsmash_results.csv')
df_results.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')